In [1]:
import os
import pandas as pd
import numpy as np
import shutil
from sklearn.model_selection import StratifiedGroupKFold

In [3]:
dirpath = os.getcwd()
new_dir = os.path.join(os.path.normpath(dirpath + os.sep + os.pardir), r'udder_video')
old_dir = os.path.join(os.path.normpath(dirpath + os.sep + os.pardir), r'udder_dcc')
new_dir_images = os.path.join(new_dir, r"depth_images")
old_dir_images = os.path.join(old_dir, r"depth_images")
new_dir_labels = os.path.join(new_dir, r"labels\class")
old_dir_labels = os.path.join(old_dir, r"depth_images")

In [4]:
image_dir_dict = {20210625:{"lab": old_dir_images},\
                  20211022: {"lab": old_dir_images},\
                  20231117:{"guilherme": os.path.join(new_dir_images, "videos_2"),\
                            "maria":os.path.join(new_dir_images, "videos_1")}}

label_dir_dict = {20210625:{"lab": old_dir_images},\
                  20211022: {"lab": old_dir_images},\
                  20231117:{"guilherme": os.path.join(new_dir_images, "videos_2"),\
                            "maria":os.path.join(new_dir_images, "videos_1")}}

group_dict = {20210625:{"lab": "A"}, \
              20211022: {"lab": "B"}, \
              20231117:{"guilherme":  "C", \
                        "maria":"D"}}

In [10]:
class_df = pd.read_csv("frame_class_list.csv")
class_df["video_file"] = ["_".join(file.split("_")[:-2]) for file in class_df.filename]
metadata_df = pd.read_csv(os.path.join(new_dir, r"video_metadata\video_metadata_20231117.csv"))
metadata_df["video_file"] = ["_".join([str(int(file.split("_")[0]))]+file.split("_")[1:]).replace(".bag", "") for file in metadata_df.filename]
metadata_df = metadata_df.drop(["time", "size", "filename"],  axis=1).drop_duplicates()
new_df = pd.merge(class_df, metadata_df, on = ["cow", "video_file"], how = "left")
new_df = new_df[["cow","frame_class", "filename", "date", "computer"]]

old_frames_df = pd.read_csv(os.path.join(old_dir, "class_filenames.csv"))
old_frames_df["filename"] = [file.replace(".tif", "") for file in old_frames_df.filename]
old_frames_df["date"] = [int(rd) for rd in old_frames_df["round"]]
old_frames_df["computer"] = "lab"
old_frames_df = old_frames_df.rename({"cowID": "cow"} , axis = 1)
old_frames_df["frame_class"] = [1 if frame == "good" else 0 if frame == "empty" else 3 for frame in old_frames_df.category]

old_df = old_frames_df[old_frames_df.frame_class != 3][["cow","frame_class", "filename", "date", "computer"]]

In [12]:
allframes_df = pd.concat([new_df, old_df], axis = 0, ignore_index = True)
grouped_df = allframes_df[["date", "computer", "cow", "frame_class"]].groupby(["date", "computer", "cow"]).agg(["count", "sum"]).reset_index()
grouped_df.columns = ["_".join(name) if len(name[1]) >1 else name[0] for name in grouped_df.columns]
grouped_df["frame_class_diff"] = grouped_df.frame_class_count - grouped_df.frame_class_sum
grouped_df[["date", "computer", "frame_class_sum","frame_class_diff"]].groupby(["date", "computer"]).agg(["count", "min", "max", "median", "mean"])

frame_class_sum                                \
                             count  min   max median        mean   
date     computer                                                  
20210625 lab                    29    5   258   96.0  104.517241   
20211022 lab                    34   44   282  133.0  127.794118   
20231117 guilherme              25  160   799  559.0  508.240000   
         maria                  25   43  1376  500.0  541.360000   

                   frame_class_diff                               
                              count min   max median        mean  
date     computer                                                 
20210625 lab                     29   0   891    0.0   56.379310  
20211022 lab                     34   0   391    0.0   12.794118  
20231117 guilherme               25   0   631   66.0  144.440000  
         maria                   25   0  1405  136.0  292.000000

In [13]:
filtered_frames_df = pd.DataFrame(columns = ["cow", "group", "frame_class","date", "computer","filename"])
thres = 200
for cow in grouped_df.cow:
    date = grouped_df[grouped_df.cow == cow].date.values[0]
    computer = grouped_df[grouped_df.cow == cow].computer.values[0]
    group = group_dict[date][computer]
    for frame_class in [0,1]:
        cow_frames = list(allframes_df[(allframes_df.cow == cow) & (allframes_df.frame_class == frame_class)].filename)
        num = len(cow_frames)
        
        if num > 0:
            np.random.seed(5)
            np.random.shuffle(cow_frames)
            if num > thres:
                selected_frames = cow_frames[:thres] 
                temp_df = pd.DataFrame([[cow, group, frame_class, date, computer]]*thres, columns = ["cow", "group", "frame_class","date", "computer"], index = range(thres))
                temp_df["filename"] = selected_frames
            else:
                temp_df = pd.DataFrame([[cow, group, frame_class, date, computer]]*num, columns = ["cow", "group", "frame_class","date", "computer"], index = range(num))
                temp_df["filename"] = cow_frames

            filtered_frames_df = pd.concat([filtered_frames_df, temp_df], axis = 0, ignore_index = True)

In [17]:
from sklearn.model_selection import StratifiedGroupKFold

In [41]:
test_list = []
fold_list = []
for group in ["D"]:
    x = np.array(filtered_frames_df[filtered_frames_df.group == group]["filename"])
    group_cows = np.array(list(filtered_frames_df[filtered_frames_df.group == group]["cow"]))
    group_class = np.array(list(filtered_frames_df[filtered_frames_df.group == group]["frame_class"]))
    sgkf = StratifiedGroupKFold(n_splits=5)
    sgkf.get_n_splits(x, group_class)
    for i, (train_index, test_index) in enumerate(sgkf.split(x, group_class, group_cows)):
        test_cows = np.unique(group_cows[test_index])
        test_list.extend(list(test_cows))
        fold_list.extend([i]*len(test_cows))

In [43]:
len(test_list)

25

In [42]:
len(test_list)

25

In [38]:
len(test_list)

29

In [40]:
len(test_list)

34